# AqSolDB: A curated aqueous solubility dataset
> ## Section 3, Test 5 new Molecules

# **summary from Section 2**:

### The best-performing model was **Extra Trees**, with the lowest error and highest explanatory power:

> MAE = 0.720

> RMSE = 1.059

> R² = 0.793


This means Extra Trees predicts solubility with an average error of about 0.72 logS units and explains approximately 79.3% of the variation in solubility.

LightGBM and Random Forest performed very similarly, with R² values of 0.787 and 0.784, respectively. This shows that tree-based ensemble models are the strongest approach for this dataset.

XGBoost and SVR also performed well, but slightly below Extra Trees, LightGBM, and Random Forest.

Linear models such as Elastic Net, Linear Regression, and Partial Least Squares performed much worse, with R² around 0.48–0.50. This confirms that aqueous solubility is not explained well by simple linear relationships.

In [ ]:
# In VS Code terminal:   pip install rdkit

In [73]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, GraphDescriptors
import pandas as pd

In [74]:
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    descriptors = {
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Crippen.MolLogP(mol),
        "MolMR": Crippen.MolMR(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "NumHAcceptors": rdMolDescriptors.CalcNumHBA(mol),
        "NumHDonors": rdMolDescriptors.CalcNumHBD(mol),
        "NumHeteroatoms": rdMolDescriptors.CalcNumHeteroatoms(mol),
        "NumRotatableBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "NumValenceElectrons": Descriptors.NumValenceElectrons(mol),
        "NumAromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "NumSaturatedRings": rdMolDescriptors.CalcNumSaturatedRings(mol),
        "NumAliphaticRings": rdMolDescriptors.CalcNumAliphaticRings(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "LabuteASA": rdMolDescriptors.CalcLabuteASA(mol),
        "BalabanJ": GraphDescriptors.BalabanJ(mol),
        "BertzCT": GraphDescriptors.BertzCT(mol)
    }

    return descriptors

In [75]:
new_molecules = pd.DataFrame({
    "Name": [
        "Isopropyl 4-chlorobenzoate",
        "1-(4-methoxyphenyl)piperazine",
        "4-fluoro-N-(2-hydroxyethyl)benzamide",
        "4-methylsulfonylbenzamide",
        "Ethyl 2-(4-methoxyphenyl)acetate"
    ],
    "SMILES": [
        "CC(C)OC(=O)c1ccc(Cl)cc1",
        "COc1ccc(N2CCNCC2)cc1",
        "O=C(NCCO)c1ccc(F)cc1",
        "CS(=O)(=O)c1ccc(C(N)=O)cc1",
        "CCOC(=O)Cc1ccc(OC)cc1"
    ]
})

In [76]:
descriptor_data = []

for smiles in new_molecules["SMILES"]:
    descriptor_data.append(calculate_descriptors(smiles))

new_descriptor_df = pd.DataFrame(descriptor_data)

new_descriptor_df

,MolWt,MolLogP,MolMR,HeavyAtomCount,NumHAcceptors,NumHDonors,NumHeteroatoms,NumRotatableBonds,NumValenceElectrons,NumAromaticRings,NumSaturatedRings,NumAliphaticRings,RingCount,TPSA,LabuteASA,BalabanJ,BertzCT
0,198.649,2.9052,52.0035,13,2,0,3,2,70,1,0,0,1,26.30,82.469505,2.721057,290.297985
1,192.262,1.1048,58.0767,14,3,1,3,2,76,1,1,1,2,24.50,84.673524,2.070894,277.843935
2,183.182,0.5478,45.8190,13,2,2,4,3,70,1,0,0,1,49.33,75.197383,2.611171,284.171105
3,199.231,0.1890,48.0697,13,3,1,5,2,70,1,0,0,1,77.23,76.288099,3.155269,419.207291
4,194.230,1.8008,53.3310,14,3,0,3,4,76,1,0,0,1,35.53,83.644726,2.549829,289.905883


In [77]:
new_X = new_descriptor_df[descriptor_columns]

In [78]:
extra_trees_model

,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [79]:
new_predictions = extra_trees_model.predict(new_X)

In [80]:
new_molecules["Predicted_Solubility"] = new_predictions

new_molecules

,Name,SMILES,Predicted_Solubility
0,Isopropyl 4-chlorobenzoate,CC(C)OC(=O)c1ccc(Cl)cc1,-2.996798
1,1-(4-methoxyphenyl)piperazine,COc1ccc(N2CCNCC2)cc1,-1.341471
2,4-fluoro-N-(2-hydroxyethyl)benzamide,O=C(NCCO)c1ccc(F)cc1,-1.348480
3,4-methylsulfonylbenzamide,CS(=O)(=O)c1ccc(C(N)=O)cc1,-1.558082
4,Ethyl 2-(4-methoxyphenyl)acetate,CCOC(=O)Cc1ccc(OC)cc1,-2.489795


In [81]:
def classify_solubility_risk(logs):
    if logs > -2:
        return "Good solubility"
    elif logs > -4:
        return "Moderate solubility risk"
    elif logs > -6:
        return "Poor solubility"
    else:
        return "Very poor solubility"

new_molecules["Solubility_Risk"] = new_molecules["Predicted_Solubility"].apply(classify_solubility_risk)

new_molecules

,Name,SMILES,Predicted_Solubility,Solubility_Risk
0,Isopropyl 4-chlorobenzoate,CC(C)OC(=O)c1ccc(Cl)cc1,-2.996798,Moderate solubility risk
1,1-(4-methoxyphenyl)piperazine,COc1ccc(N2CCNCC2)cc1,-1.341471,Good solubility
2,4-fluoro-N-(2-hydroxyethyl)benzamide,O=C(NCCO)c1ccc(F)cc1,-1.348480,Good solubility
3,4-methylsulfonylbenzamide,CS(=O)(=O)c1ccc(C(N)=O)cc1,-1.558082,Good solubility
4,Ethyl 2-(4-methoxyphenyl)acetate,CCOC(=O)Cc1ccc(OC)cc1,-2.489795,Moderate solubility risk


>> ## ***Author:*** Sam Pharmed [Kaggle](https://www.kaggle.com/sampharmed), [GitHub](https://github.com/Samb-pharmed)  
>> **Date:** 19 April 2026  
>> **Dataset:** [AqSolDB: A curated aqueous solubility dataset](https://www.kaggle.com/datasets/sorkun/aqsoldb-a-curated-aqueous-solubility-dataset)   